In [1]:
import ROOT

In [2]:
file_path = "../root_ML/merged_ML-600to800.root"
file = ROOT.TFile(file_path)
tree = file.Get("jetTree")

n_entries = tree.GetEntries()
print(f"Number of entries in the tree: {n_entries}")

Number of entries in the tree: 5737037


In [3]:
hist_2D = ROOT.TH2F("hist_2D", "2D Histogram; qq_score; matched_score", 10, 0, 1, 10, 0, 1)
hist_2D_ch1 = ROOT.TH2F("hist_2D_purity", "2D Histogram (Purity); qq_score; matched_score", 10, 0, 1, 10, 0, 1)

In [4]:
for event in range(n_entries):
    tree.GetEntry(event)
    for jet_i in range(len(tree.jet_pt)):
        qq_score = tree.lund_ML_qq[jet_i]
        matched_score = tree.lund_ML_matched[jet_i]

        channel2 = tree.lund_secondary_idx_sd[jet_i]

        hist_2D.Fill(qq_score, matched_score)
        if channel2 == 1:
            hist_2D_ch1.Fill(qq_score, matched_score)

In [5]:
for x_bin in range(1, hist_2D.GetNbinsX() + 1):
    for y_bin in range(1, hist_2D.GetNbinsY() + 1):
        total = hist_2D.GetBinContent(x_bin, y_bin)
        purity = hist_2D_ch1.GetBinContent(x_bin, y_bin)
        
        if total > 0:  # Avoid division by zero
            hist_2D_ch1.SetBinContent(x_bin, y_bin, purity / total)
        else:
            hist_2D_ch1.SetBinContent(x_bin, y_bin, 0)  # Set purity to 0 if no entries

In [6]:
canvas = ROOT.TCanvas("canvas", "Canvas", 800, 600)
hist_2D.GetXaxis().SetTitle("qq_score")
hist_2D.GetYaxis().SetTitle("matched_score")

#Set axis ranges
hist_2D.GetXaxis().SetRangeUser(0, 1)
hist_2D.GetYaxis().SetRangeUser(0, 1)


hist_2D.SetTitle("2D Histogram of qq_score vs matched_score")
hist_2D.SetStats(0)  # Disable statistics box
hist_2D_ch1.Draw("COLZ")  # Draw purity histogram on top
hist_2D.Draw("TEXT SAME")
canvas.Draw()

In [15]:
hist_dpsi = ROOT.TH1F("hist_dpsi", "Histogram of dpsi; dpsi; Entries", 20, 0, 3.14)

hist2D_z1_matched = ROOT.TH2F("hist2D_z1", "2D Histogram of z1 vs matched_score; z1; matched_score", 10, 0, 0.5, 10, 0, 1)
hist2D_z2_matched = ROOT.TH2F("hist2D_z2", "2D Histogram of z2 vs matched_score; z2; matched_score", 10, 0, 0.5, 10, 0, 1)

hist2D_z1_qq = ROOT.TH2F("hist2D_z1_qq", "2D Histogram of z1 vs qq_score; z1; qq_score", 10, 0, 0.5, 10, 0, 1)
hist2D_z2_qq = ROOT.TH2F("hist2D_z2_qq", "2D Histogram of z2 vs qq_score; z2; qq_score", 10, 0, 0.5, 10, 0, 1)

hist2D_pt_matched = ROOT.TH2F("hist2D_pt_matched", "2D Histogram of pt vs matched_score; pt; matched_score", 10, 700, 2000, 10, 0, 1)
hist2D_pt_qq = ROOT.TH2F("hist2D_pt_qq", "2D Histogram of pt vs qq_score; pt; qq_score", 10, 700, 2000, 10, 0, 1)

Warning in <TFile::Append>: Replacing existing TH1: hist_dpsi (Potential memory leak).
Warning in <TFile::Append>: Replacing existing TH1: hist2D_z1 (Potential memory leak).
Warning in <TFile::Append>: Replacing existing TH1: hist2D_z2 (Potential memory leak).
Warning in <TFile::Append>: Replacing existing TH1: hist2D_z1_qq (Potential memory leak).
Warning in <TFile::Append>: Replacing existing TH1: hist2D_z2_qq (Potential memory leak).
Warning in <TFile::Append>: Replacing existing TH1: hist2D_pt_matched (Potential memory leak).
Warning in <TFile::Append>: Replacing existing TH1: hist2D_pt_qq (Potential memory leak).


In [16]:
min_qq_score = 0.6
min_matched_score = 0.9
for event in range(n_entries):
    tree.GetEntry(event)
    for jet_i in range(len(tree.jet_pt)):
        qq_score = tree.lund_ML_qq[jet_i]
        matched_score = tree.lund_ML_matched[jet_i]

        idx1 = int(tree.lund_max_kt_sd[jet_i])
        idx2 = int(tree.lund_max_kt_secondary_sd[jet_i])
        
        if idx1 < 0 or idx2 < 0:
            continue  # Skip this jet if indices are invalid

        z1 = tree.lund_z_sd[jet_i][idx1]
        z2 = tree.lund_z_secondary_sd[jet_i][idx2]

        hist2D_z1_matched.Fill(z1, matched_score)
        hist2D_z2_matched.Fill(z2, matched_score)

        hist2D_z1_qq.Fill(z1, qq_score)
        hist2D_z2_qq.Fill(z2, qq_score)
        
        hist2D_pt_matched.Fill(tree.jet_pt[jet_i], matched_score)
        hist2D_pt_qq.Fill(tree.jet_pt[jet_i], qq_score)
        
        if qq_score > min_qq_score and matched_score > min_matched_score:
            dpsi = tree.lund_psi12_sd[jet_i]
            hist_dpsi.Fill(dpsi)

In [17]:
canvas_dpsi = ROOT.TCanvas("canvas_dpsi", "Canvas for dpsi", 800, 600)
hist_dpsi.SetTitle(f"Histogram of dpsi for qq_score > {min_qq_score} and matched_score > {min_matched_score}")
hist_dpsi.SetStats(0)  # Disable statistics box
hist_dpsi.GetXaxis().SetTitle("dpsi")
hist_dpsi.GetYaxis().SetTitle("Entries")
hist_dpsi.Draw()
canvas_dpsi.Draw()

In [18]:
import numpy as np
#calculate <cos(2dpsi)> of hist_dpsi

cos_2dpsi_sum = 0
total_entries = 0

for x_bin in range(1, hist_dpsi.GetNbinsX() + 1):
    dpsi = hist_dpsi.GetBinCenter(x_bin)
    entries = hist_dpsi.GetBinContent(x_bin)
    cos_2dpsi_sum += entries * np.cos(2 * dpsi)
    total_entries += entries

print(f"<cos(2dpsi)> = {cos_2dpsi_sum / total_entries if total_entries > 0 else 0}")

<cos(2dpsi)> = -0.29751881183396217


In [19]:
canvas_z1_matched = ROOT.TCanvas("canvas_z1_matched", "Canvas for z1 vs matched_score", 800, 600)
hist2D_z1_matched.SetTitle("2D Histogram of z1 vs matched_score")
hist2D_z1_matched.SetStats(0)  # Disable statistics box
hist2D_z1_matched.GetXaxis().SetTitle("z1")
hist2D_z1_matched.GetYaxis().SetTitle("matched_score")
hist2D_z1_matched.Draw("COLZ")
canvas_z1_matched.Draw()

In [20]:
canvas_z2_matched = ROOT.TCanvas("canvas_z2_matched", "Canvas for z2 vs matched_score", 800, 600)
hist2D_z2_matched.SetTitle("2D Histogram of z2 vs matched_score")
hist2D_z2_matched.SetStats(0)  # Disable statistics box
hist2D_z2_matched.GetXaxis().SetTitle("z2")
hist2D_z2_matched.GetYaxis().SetTitle("matched_score")
hist2D_z2_matched.Draw("COLZ")
canvas_z2_matched.Draw()

In [21]:
canvas_z1_qq = ROOT.TCanvas("canvas_z1_qq", "Canvas for z1 vs qq_score", 800, 600)
hist2D_z1_qq.SetTitle("2D Histogram of z1 vs qq_score")
hist2D_z1_qq.SetStats(0)  # Disable statistics box
hist2D_z1_qq.GetXaxis().SetTitle("z1")
hist2D_z1_qq.GetYaxis().SetTitle("qq_score")
hist2D_z1_qq.Draw("COLZ")
canvas_z1_qq.Draw()

In [22]:
canvas_z2_qq = ROOT.TCanvas("canvas_z2_qq", "Canvas for z2 vs qq_score", 800, 600)
hist2D_z2_qq.SetTitle("2D Histogram of z2 vs qq_score")
hist2D_z2_qq.SetStats(0)  # Disable statistics box
hist2D_z2_qq.GetXaxis().SetTitle("z2")
hist2D_z2_qq.GetYaxis().SetTitle("qq_score")
hist2D_z2_qq.Draw("COLZ")
canvas_z2_qq.Draw()

In [23]:
canvas_pt_matched = ROOT.TCanvas("canvas_pt_matched", "Canvas for pt vs matched_score", 800, 600)
hist2D_pt_matched.SetTitle("2D Histogram of pt vs matched_score")
hist2D_pt_matched.SetStats(0)  # Disable statistics box
hist2D_pt_matched.GetXaxis().SetTitle("pt")
hist2D_pt_matched.GetYaxis().SetTitle("matched_score")
hist2D_pt_matched.Draw("COLZ")
canvas_pt_matched.Draw()

In [24]:
canvas_pt_qq = ROOT.TCanvas("canvas_pt_qq", "Canvas for pt vs qq_score", 800, 600)
hist2D_pt_qq.SetTitle("2D Histogram of pt vs qq_score")
hist2D_pt_qq.SetStats(0)  # Disable statistics box
hist2D_pt_qq.GetXaxis().SetTitle("pt")
hist2D_pt_qq.GetYaxis().SetTitle("qq_score")
hist2D_pt_qq.Draw("COLZ")
canvas_pt_qq.Draw()